# 01. CORD v2 데이터셋 탐색 (EDA)

**목표**
- CORD v2 로드 및 구조 파악
- 이미지 크기, 필드 분포, 라벨 통계 분석
- Florence-2 타겟 시퀀스 변환 검증

**실행 환경**: Kaggle / Google Colab

In [ ]:
# 필요 라이브러리 설치 (Kaggle/Colab 첫 실행 시)
!pip install -q datasets transformers matplotlib seaborn pandas

In [ ]:
import json
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import load_dataset

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'

## 1. 데이터셋 로드

In [ ]:
dataset = load_dataset("naver-clova-ix/cord-v2")
print(dataset)
# 예상: train 800 / validation 100 / test 100

In [ ]:
# 샘플 1개 구조 확인
sample = dataset["train"][0]
print("Keys:", sample.keys())
print("Image size:", sample["image"].size)
print("\nground_truth (raw):")
gt = json.loads(sample["ground_truth"])
print(json.dumps(gt, indent=2, ensure_ascii=False))

## 2. 이미지 크기 분포

In [ ]:
widths, heights = [], []
for sample in dataset["train"]:
    w, h = sample["image"].size
    widths.append(w)
    heights.append(h)

fig, axes = plt.subplots(1, 2)
axes[0].hist(widths, bins=30, color='steelblue')
axes[0].set_title("Width Distribution")
axes[0].set_xlabel("pixels")

axes[1].hist(heights, bins=30, color='coral')
axes[1].set_title("Height Distribution")
axes[1].set_xlabel("pixels")

plt.tight_layout()
plt.show()

print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {sum(widths)/len(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {sum(heights)/len(heights):.0f}")

## 3. 필드 분포 분석

In [ ]:
field_counter = Counter()
menu_lengths = []

for sample in dataset["train"]:
    gt = json.loads(sample["ground_truth"])["gt_parse"]

    for key in gt.keys():
        field_counter[key] += 1

    if "menu" in gt:
        menu_lengths.append(len(gt["menu"]))

# 필드 출현 빈도
df_fields = pd.DataFrame(field_counter.items(), columns=["field", "count"]).sort_values("count", ascending=False)

plt.figure()
sns.barplot(data=df_fields, x="count", y="field", palette="Blues_r")
plt.title("Top-level Field Distribution (train set)")
plt.tight_layout()
plt.show()

print(df_fields.to_string(index=False))

In [ ]:
# 영수증당 메뉴 아이템 수 분포
plt.figure()
plt.hist(menu_lengths, bins=range(1, max(menu_lengths)+2), color='mediumpurple', edgecolor='white')
plt.title("Menu Items per Receipt")
plt.xlabel("# items")
plt.ylabel("count")
plt.tight_layout()
plt.show()

print(f"메뉴 아이템 수 — min: {min(menu_lengths)}, max: {max(menu_lengths)}, mean: {sum(menu_lengths)/len(menu_lengths):.1f}")

## 4. 타겟 시퀀스 변환 검증

In [ ]:
import sys
sys.path.append("..")  # Kaggle: repo를 /kaggle/input/에 마운트 후 경로 조정

from src.data.dataset import cord_to_target_sequence

# 샘플 5개 변환 결과 확인
for i in range(5):
    sample = dataset["train"][i]
    gt = json.loads(sample["ground_truth"])
    target_seq = cord_to_target_sequence(gt)
    print(f"--- Sample {i} ---")
    print(target_seq[:300], "...\n" if len(target_seq) > 300 else "\n")

In [ ]:
# 타겟 시퀀스 길이 분포
seq_lengths = []
for sample in dataset["train"]:
    gt = json.loads(sample["ground_truth"])
    seq = cord_to_target_sequence(gt)
    seq_lengths.append(len(seq.split()))  # 단어 기준 (토큰 기준은 processor 필요)

plt.figure()
plt.hist(seq_lengths, bins=30, color='teal')
plt.axvline(512, color='red', linestyle='--', label='max_length=512')
plt.title("Target Sequence Length Distribution")
plt.xlabel("# words")
plt.legend()
plt.tight_layout()
plt.show()

over_512 = sum(1 for l in seq_lengths if l > 512)
print(f"max_length=512 초과 샘플: {over_512}/{len(seq_lengths)} ({over_512/len(seq_lengths)*100:.1f}%)")

## 5. 샘플 이미지 시각화

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, ax in enumerate(axes.flat):
    sample = dataset["train"][i]
    gt = json.loads(sample["ground_truth"])
    seq = cord_to_target_sequence(gt)

    ax.imshow(sample["image"])
    ax.set_title(f"Sample {i}\n{seq[:80]}...", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. EDA 요약 → EXPERIMENTS.md에 기록

```
아래 결과를 docs/EXPERIMENTS.md에 복사하세요.
```

In [ ]:
print("=== CORD v2 EDA 요약 ===")
print(f"Train: {len(dataset['train'])}개 / Validation: {len(dataset['validation'])}개 / Test: {len(dataset['test'])}개")
print(f"이미지 크기: W {min(widths)}~{max(widths)}px / H {min(heights)}~{max(heights)}px")
print(f"평균 메뉴 아이템 수: {sum(menu_lengths)/len(menu_lengths):.1f}개")
print(f"타겟 시퀀스 max_length=512 초과: {over_512}개")
print("\n주요 필드:")
print(df_fields.to_string(index=False))